In [1]:
# Install compatible versions for Spark + Delta Lake on Colab
!pip install pyspark==3.5.0 delta-spark==3.0.0 pandas --quiet

print("Dependencies installed. Ready to initialize Spark with Delta Lake.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Dependencies installed. Ready to initialize Spark with Delta Lake.


In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession with Delta Lake properly attached
builder = SparkSession.builder \
    .appName("FeatureStore_DeltaLake") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "4")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark


In [3]:
import os

# Explicit, interview-defensible structure
os.makedirs("data/raw", exist_ok=True)        # source-of-truth CSVs
os.makedirs("delta/raw_table", exist_ok=True) # immutable raw Delta table
os.makedirs("delta/features", exist_ok=True)  # versioned feature store

print("Project directory structure created:")
for root, dirs, files in os.walk(".", topdown=True):
    if root.count(os.sep) <= 2:
        print(root)


Project directory structure created:
.
./.config
./.config/logs
./.config/configurations
./data
./data/raw
./delta
./delta/raw_table
./delta/features
./sample_data


In [4]:
import pandas as pd

# Simulated source-system events (realistic for feature engineering)
raw_events = pd.DataFrame({
    "user_id": [101, 101, 102, 103, 102, 101, 104, 103],
    "event_type": [
        "purchase", "purchase", "refund",
        "purchase", "purchase", "refund",
        "purchase", "purchase"
    ],
    "amount": [250.0, 120.0, -50.0, 300.0, 180.0, -120.0, 500.0, 220.0],
    "event_ts": [
        "2024-01-01 10:15:00",
        "2024-01-03 14:20:00",
        "2024-01-04 09:05:00",
        "2024-01-05 18:45:00",
        "2024-01-06 11:30:00",
        "2024-01-07 16:10:00",
        "2024-01-08 12:00:00",
        "2024-01-09 19:25:00"
    ]
})

raw_csv_path = "data/raw/events.csv"
raw_events.to_csv(raw_csv_path, index=False)

print(f"Raw CSV written to {raw_csv_path}")
raw_events.head()


Raw CSV written to data/raw/events.csv


,user_id,event_type,amount,event_ts
0,101,purchase,250.0,2024-01-01 10:15:00
1,101,purchase,120.0,2024-01-03 14:20:00
2,102,refund,-50.0,2024-01-04 09:05:00
3,103,purchase,300.0,2024-01-05 18:45:00
4,102,purchase,180.0,2024-01-06 11:30:00


In [5]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DoubleType, TimestampType
)

# Explicit schema to avoid inference issues and ensure reproducibility
raw_schema = StructType([
    StructField("user_id", IntegerType(), nullable=False),
    StructField("event_type", StringType(), nullable=False),
    StructField("amount", DoubleType(), nullable=False),
    StructField("event_ts", TimestampType(), nullable=False),
])

raw_df = spark.read \
    .schema(raw_schema) \
    .option("header", True) \
    .csv("data/raw/events.csv")

# Basic validation
raw_df.printSchema()
raw_df.show(5, truncate=False)


root
 |-- user_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_ts: timestamp (nullable = true)

+-------+----------+------+-------------------+
|user_id|event_type|amount|event_ts           |
+-------+----------+------+-------------------+
|101    |purchase  |250.0 |2024-01-01 10:15:00|
|101    |purchase  |120.0 |2024-01-03 14:20:00|
|102    |refund    |-50.0 |2024-01-04 09:05:00|
|103    |purchase  |300.0 |2024-01-05 18:45:00|
|102    |purchase  |180.0 |2024-01-06 11:30:00|
+-------+----------+------+-------------------+
only showing top 5 rows



In [6]:
# Write raw data to Delta as an immutable source table
raw_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("delta/raw_table")

print("Raw data written to Delta (raw_table). Versioning initialized.")


Raw data written to Delta (raw_table). Versioning initialized.


In [7]:
from delta.tables import DeltaTable

# Read raw Delta table
raw_delta_df = spark.read.format("delta").load("delta/raw_table")

# Validate data
raw_delta_df.show(5, truncate=False)

# Inspect Delta version history (time travel metadata)
raw_delta_table = DeltaTable.forPath(spark, "delta/raw_table")
raw_delta_table.history().show(truncate=False)


+-------+----------+------+-------------------+
|user_id|event_type|amount|event_ts           |
+-------+----------+------+-------------------+
|101    |purchase  |250.0 |2024-01-01 10:15:00|
|101    |purchase  |120.0 |2024-01-03 14:20:00|
|102    |refund    |-50.0 |2024-01-04 09:05:00|
|103    |purchase  |300.0 |2024-01-05 18:45:00|
|102    |purchase  |180.0 |2024-01-06 11:30:00|
+-------+----------+------+-------------------+
only showing top 5 rows

+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                        

In [8]:
from pyspark.sql import functions as F

# Derive user-level features from raw events
features_df_v1 = raw_delta_df.groupBy("user_id").agg(
    F.count("*").alias("event_count"),
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchase_count"),
    F.sum(F.when(F.col("event_type") == "refund", 1).otherwise(0)).alias("refund_count"),
    F.max("event_ts").alias("last_event_ts")
)

features_df_v1.show(truncate=False)


+-------+-----------+------------+-----------------+--------------+------------+-------------------+
|user_id|event_count|total_amount|avg_amount       |purchase_count|refund_count|last_event_ts      |
+-------+-----------+------------+-----------------+--------------+------------+-------------------+
|101    |3          |250.0       |83.33333333333333|2             |1           |2024-01-07 16:10:00|
|102    |2          |130.0       |65.0             |1             |1           |2024-01-06 11:30:00|
|104    |1          |500.0       |500.0            |1             |0           |2024-01-08 12:00:00|
|103    |2          |520.0       |260.0            |2             |0           |2024-01-09 19:25:00|
+-------+-----------+------------+-----------------+--------------+------------+-------------------+



In [9]:
# Write initial feature set to Delta feature store (Version 0)
features_df_v1.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("delta/features")

print("Feature store written to Delta (features). Version 0 created.")


Feature store written to Delta (features). Version 0 created.


In [10]:
from delta.tables import DeltaTable

# Read current feature store
features_current = spark.read.format("delta").load("delta/features")
features_current.show(truncate=False)

# Inspect Delta history to confirm versioning
features_table = DeltaTable.forPath(spark, "delta/features")
features_table.history().show(truncate=False)


+-------+-----------+------------+-----------------+--------------+------------+-------------------+
|user_id|event_count|total_amount|avg_amount       |purchase_count|refund_count|last_event_ts      |
+-------+-----------+------------+-----------------+--------------+------------+-------------------+
|101    |3          |250.0       |83.33333333333333|2             |1           |2024-01-07 16:10:00|
|102    |2          |130.0       |65.0             |1             |1           |2024-01-06 11:30:00|
|104    |1          |500.0       |500.0            |1             |0           |2024-01-08 12:00:00|
|103    |2          |520.0       |260.0            |2             |0           |2024-01-09 19:25:00|
+-------+-----------+------------+-----------------+--------------+------------+-------------------+

+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------

In [11]:
from pyspark.sql import functions as F

# Evolve features: add stronger ML-oriented signals
features_df_v2 = raw_delta_df.groupBy("user_id").agg(
    F.count("*").alias("event_count"),
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.max("amount").alias("max_amount"),
    F.min("amount").alias("min_amount"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchase_count"),
    F.sum(F.when(F.col("event_type") == "refund", 1).otherwise(0)).alias("refund_count"),
    F.max("event_ts").alias("last_event_ts"),
    F.datediff(F.current_timestamp(), F.max("event_ts")).alias("days_since_last_event")
)

features_df_v2.show(truncate=False)


+-------+-----------+------------+-----------------+----------+----------+--------------+------------+-------------------+---------------------+
|user_id|event_count|total_amount|avg_amount       |max_amount|min_amount|purchase_count|refund_count|last_event_ts      |days_since_last_event|
+-------+-----------+------------+-----------------+----------+----------+--------------+------------+-------------------+---------------------+
|101    |3          |250.0       |83.33333333333333|250.0     |-120.0    |2             |1           |2024-01-07 16:10:00|725                  |
|102    |2          |130.0       |65.0             |180.0     |-50.0     |1             |1           |2024-01-06 11:30:00|726                  |
|104    |1          |500.0       |500.0            |500.0     |500.0     |1             |0           |2024-01-08 12:00:00|724                  |
|103    |2          |520.0       |260.0            |300.0     |220.0     |2             |0           |2024-01-09 19:25:00|723     

In [12]:
# Overwrite feature store with evolved features (creates Version 1)
features_df_v2.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("delta/features")

print("Feature store updated. Version 1 created.")


Feature store updated. Version 1 created.


In [13]:
# Read Feature Store Version 0 using Delta time travel
features_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("delta/features")

print("Feature Store - Version 0")
features_v0.show(truncate=False)

# Read latest Feature Store version (Version 1)
features_latest = spark.read.format("delta").load("delta/features")

print("Feature Store - Latest Version")
features_latest.show(truncate=False)


Feature Store - Version 0
+-------+-----------+------------+-----------------+--------------+------------+-------------------+
|user_id|event_count|total_amount|avg_amount       |purchase_count|refund_count|last_event_ts      |
+-------+-----------+------------+-----------------+--------------+------------+-------------------+
|101    |3          |250.0       |83.33333333333333|2             |1           |2024-01-07 16:10:00|
|102    |2          |130.0       |65.0             |1             |1           |2024-01-06 11:30:00|
|104    |1          |500.0       |500.0            |1             |0           |2024-01-08 12:00:00|
|103    |2          |520.0       |260.0            |2             |0           |2024-01-09 19:25:00|
+-------+-----------+------------+-----------------+--------------+------------+-------------------+

Feature Store - Latest Version
+-------+-----------+------------+-----------------+----------+----------+--------------+------------+-------------------+------------

In [14]:
from delta.tables import DeltaTable

# Inspect full history of the feature store
features_table = DeltaTable.forPath(spark, "delta/features")
features_table.history().show(truncate=False)


+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|1      |2026-01-01 09:34:48.487|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |0          |Serializable  |false        |{numFiles -> 1, nu

**Short Note**

**(Course Learner Perspective)**

In this project, I built a simple feature store using PySpark and Delta Lake to understand how feature versioning and reproducibility are handled in real-world machine learning systems. The project starts with ingesting raw CSV event data into an immutable Delta table using an explicit schema, ensuring data consistency and reliability.

I then performed feature engineering using Spark aggregations to generate meaningful user-level features such as transaction counts, total and average amounts, behavioral signals, and recency-based features. These derived features were stored in a Delta table acting as a feature store.

By evolving the feature logic and overwriting the feature table, Delta Lake automatically created multiple versions. Using Delta Lake’s time travel, I was able to query historical feature versions, demonstrating how older feature definitions can be retrieved for model reproducibility and debugging.

This project helped me understand core feature store concepts, Delta Lake ACID guarantees, version control for data, and the importance of reproducible features in machine learning pipelines.

**Conclusion**

This project demonstrates how Delta Lake can be used to build a simple yet effective feature store with built-in versioning and time travel capabilities. By separating raw data ingestion from feature computation and leveraging Delta Lake’s transaction log, the system supports safe feature evolution without losing historical data.

The ability to retrieve past feature versions ensures reproducible model training, reliable backfills, and easier debugging, which are critical requirements in production ML systems. Overall, this project provides a strong foundation for understanding feature stores and modern data engineering practices using Spark and Delta Lake.